In [ ]:
import os
import re
import sys
import json
import math
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

In [ ]:
PROJECT_ROOT = Path.home() / "si630_nlp_project"

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
INTERIM_DATA_DIR = DATA_DIR / "interim"

MODELS_DIR = PROJECT_ROOT / "models"
EMBEDDINGS_DIR = MODELS_DIR / "embeddings"
BASELINE_MODELS_DIR = MODELS_DIR / "baselines"

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
REPORTS_DIR = OUTPUTS_DIR / "reports"
TABLE_DIR = OUTPUTS_DIR / "tables"

for p in [INTERIM_DATA_DIR, BASELINE_MODELS_DIR, REPORTS_DIR, TABLE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)
print(PROCESSED_DATA_DIR)
print(EMBEDDINGS_DIR)

In [ ]:
DATASET_TAG = "large_lite"
EXPERIMENT_TAG = "han_v1"

TEXT_COL = "full_text"
LABEL_COL = "label_5d"

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DEBUG_SMALL = False

TRAIN_SUBSAMPLE_DOCS = None

# document truncation
MAX_SENTENCES = 30
MAX_WORDS_PER_SENTENCE = 30

# vocab
MIN_FREQ = 5
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"

# model
EMBED_DIM = 100
WORD_HIDDEN_SIZE = 96
SENT_HIDDEN_SIZE = 96
DROPOUT = 0.4
NUM_CLASSES = 2

# training
BATCH_SIZE = 16
NUM_EPOCHS = 3
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
EARLY_STOPPING_PATIENCE = 2

USE_WORD2VEC_INIT = True
WORD2VEC_KV_PATH = EMBEDDINGS_DIR / "word2vec_large_lite.kv"

print("Using device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Word2Vec init path:", WORD2VEC_KV_PATH)
print("Word2Vec init exists:", WORD2VEC_KV_PATH.exists())

In [ ]:
RUN_TAG = "debug_small" if DEBUG_SMALL else ("subsample" if TRAIN_SUBSAMPLE_DOCS is not None else "full")

MODEL_PATH = BASELINE_MODELS_DIR / f"han_{DATASET_TAG}_{EXPERIMENT_TAG}_{RUN_TAG}.pt"
VOCAB_PATH = BASELINE_MODELS_DIR / f"han_{DATASET_TAG}_{EXPERIMENT_TAG}_{RUN_TAG}_vocab.json"
REPORT_PATH = REPORTS_DIR / f"han_{DATASET_TAG}_{EXPERIMENT_TAG}_{RUN_TAG}_results.json"
HISTORY_PATH = TABLE_DIR / f"han_{DATASET_TAG}_{EXPERIMENT_TAG}_{RUN_TAG}_training_history.csv"


TRAIN_CACHE_PATH = INTERIM_DATA_DIR / f"han_{DATASET_TAG}_{EXPERIMENT_TAG}_{RUN_TAG}_train_cache.npz"
VAL_CACHE_PATH = INTERIM_DATA_DIR / f"han_{DATASET_TAG}_{EXPERIMENT_TAG}_{RUN_TAG}_val_cache.npz"
TEST_CACHE_PATH = INTERIM_DATA_DIR / f"han_{DATASET_TAG}_{EXPERIMENT_TAG}_{RUN_TAG}_test_cache.npz"

print(MODEL_PATH)
print(REPORT_PATH)
print(TRAIN_CACHE_PATH)

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

In [ ]:
train_df = pd.read_parquet(PROCESSED_DATA_DIR / "train_doc_5d_large_lite.parquet")
val_df = pd.read_parquet(PROCESSED_DATA_DIR / "validation_doc_5d_large_lite.parquet")
test_df = pd.read_parquet(PROCESSED_DATA_DIR / "test_doc_5d_large_lite.parquet")

print("Original shapes:")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

if DEBUG_SMALL:
    train_df = train_df.head(500).copy()
    val_df = val_df.head(100).copy()
    test_df = test_df.head(150).copy()
    print("\nDEBUG_SMALL is ON")

if TRAIN_SUBSAMPLE_DOCS is not None:
    train_df = train_df.sample(TRAIN_SUBSAMPLE_DOCS, random_state=SEED).reset_index(drop=True)
    print(f"\nUsing TRAIN_SUBSAMPLE_DOCS = {TRAIN_SUBSAMPLE_DOCS}")

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("\nFinal shapes:")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain label distribution:")
print(train_df[LABEL_COL].value_counts(dropna=False).sort_index())

In [ ]:
_SENT_SPLIT_REGEX = re.compile(r"(?<=[\.\?\!;])\s+")

def split_into_sentences(text: str):
    text = str(text).strip()
    if not text:
        return [""]
    sents = _SENT_SPLIT_REGEX.split(text)
    sents = [s.strip() for s in sents if s.strip()]
    return sents if sents else [""]

def tokenize_words(text: str):
    return re.findall(r"[A-Za-z0-9]+", str(text).lower())

def preprocess_document(text: str):
    """
    returns: List[List[str]]
    """
    sentences = split_into_sentences(text)[:MAX_SENTENCES]
    processed = []

    for sent in sentences:
        tokens = tokenize_words(sent)[:MAX_WORDS_PER_SENTENCE]
        if not tokens:
            tokens = [UNK_TOKEN]
        processed.append(tokens)

    if not processed:
        processed = [[UNK_TOKEN]]

    return processed

In [ ]:
def build_vocab(train_texts, min_freq=5):
    counter = Counter()

    for text in tqdm(train_texts, desc="Building vocab"):
        doc = preprocess_document(text)
        for sent in doc:
            counter.update(sent)

    vocab = {
        PAD_TOKEN: 0,
        UNK_TOKEN: 1,
    }

    for token, freq in counter.items():
        if freq >= min_freq:
            vocab[token] = len(vocab)

    return vocab

vocab = build_vocab(train_df[TEXT_COL].astype(str).tolist(), min_freq=MIN_FREQ)
print("Vocab size:", len(vocab))
print(list(vocab.items())[:20])

In [ ]:
embedding_matrix = None

if USE_WORD2VEC_INIT and WORD2VEC_KV_PATH.exists():
    try:
        from gensim.models import KeyedVectors

        kv = KeyedVectors.load(str(WORD2VEC_KV_PATH))
        print("Loaded Word2Vec KV.")
        print("KV vector size:", kv.vector_size)

        if kv.vector_size != EMBED_DIM:
            print(f"Warning: EMBED_DIM={EMBED_DIM} but KV size={kv.vector_size}. "
                  f"Will reset EMBED_DIM to {kv.vector_size}.")
            EMBED_DIM = kv.vector_size

        embedding_matrix = np.random.normal(0, 0.1, size=(len(vocab), EMBED_DIM)).astype(np.float32)
        embedding_matrix[vocab[PAD_TOKEN]] = np.zeros(EMBED_DIM, dtype=np.float32)

        matched = 0
        for token, idx in vocab.items():
            if token in [PAD_TOKEN, UNK_TOKEN]:
                continue
            if token in kv:
                embedding_matrix[idx] = kv[token]
                matched += 1

        print("Matched vocab tokens in Word2Vec:", matched)
        print("Embedding matrix shape:", embedding_matrix.shape)

    except Exception as e:
        print("Could not load/use Word2Vec KV, fallback to random init.")
        print("Error:", e)
        embedding_matrix = None
else:
    print("Word2Vec init not used.")

In [ ]:
def encode_document(doc_tokens, vocab):
    """
    returns:
      doc_ids: (MAX_SENTENCES, MAX_WORDS_PER_SENTENCE)
      sent_mask: (MAX_SENTENCES,)
    """
    doc_ids = np.zeros((MAX_SENTENCES, MAX_WORDS_PER_SENTENCE), dtype=np.int64)
    sent_mask = np.zeros((MAX_SENTENCES,), dtype=np.float32)

    for i, sent in enumerate(doc_tokens[:MAX_SENTENCES]):
        sent_mask[i] = 1.0
        for j, tok in enumerate(sent[:MAX_WORDS_PER_SENTENCE]):
            doc_ids[i, j] = vocab.get(tok, vocab[UNK_TOKEN])

    return doc_ids, sent_mask

def build_encoded_arrays(df, vocab, split_name):
    all_doc_ids = []
    all_sent_masks = []
    all_labels = []

    for text, label in tqdm(
        zip(df[TEXT_COL].astype(str).tolist(), df[LABEL_COL].astype(int).tolist()),
        total=len(df),
        desc=f"Encoding {split_name}"
    ):
        doc_tokens = preprocess_document(text)
        doc_ids, sent_mask = encode_document(doc_tokens, vocab)
        all_doc_ids.append(doc_ids)
        all_sent_masks.append(sent_mask)
        all_labels.append(label)

    X_docs = np.stack(all_doc_ids).astype(np.int64)
    X_masks = np.stack(all_sent_masks).astype(np.float32)
    y = np.array(all_labels, dtype=np.int64)

    return X_docs, X_masks, y

def load_or_build_cache(df, vocab, split_name, cache_path):
    if cache_path.exists():
        print(f"Loading cached {split_name} arrays from:", cache_path)
        arr = np.load(cache_path)
        return arr["doc_ids"], arr["sent_masks"], arr["labels"]

    print(f"Building {split_name} arrays...")
    doc_ids, sent_masks, labels = build_encoded_arrays(df, vocab, split_name)

    np.savez_compressed(
        cache_path,
        doc_ids=doc_ids,
        sent_masks=sent_masks,
        labels=labels,
    )
    print(f"Saved {split_name} cache to:", cache_path)
    return doc_ids, sent_masks, labels

In [ ]:
train_doc_ids, train_sent_masks, y_train = load_or_build_cache(train_df, vocab, "train", TRAIN_CACHE_PATH)
val_doc_ids, val_sent_masks, y_val = load_or_build_cache(val_df, vocab, "validation", VAL_CACHE_PATH)
test_doc_ids, test_sent_masks, y_test = load_or_build_cache(test_df, vocab, "test", TEST_CACHE_PATH)

print("Train arrays:", train_doc_ids.shape, train_sent_masks.shape, y_train.shape)
print("Val arrays:", val_doc_ids.shape, val_sent_masks.shape, y_val.shape)
print("Test arrays:", test_doc_ids.shape, test_sent_masks.shape, y_test.shape)

In [ ]:
class HANDataset(Dataset):
    def __init__(self, doc_ids, sent_masks, labels):
        self.doc_ids = doc_ids
        self.sent_masks = sent_masks
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "doc_ids": torch.tensor(self.doc_ids[idx], dtype=torch.long),
            "sent_mask": torch.tensor(self.sent_masks[idx], dtype=torch.float),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_dataset = HANDataset(train_doc_ids, train_sent_masks, y_train)
val_dataset = HANDataset(val_doc_ids, val_sent_masks, y_val)
test_dataset = HANDataset(test_doc_ids, test_sent_masks, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

len(train_dataset), len(val_dataset), len(test_dataset)

In [ ]:
class WordAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, dropout, embedding_matrix=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        if embedding_matrix is not None:
            self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))
            print("Initialized embedding layer from Word2Vec.")

        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_size,
            batch_first=True,
            bidirectional=True,
        )
        self.attn = nn.Linear(hidden_size * 2, hidden_size * 2)
        self.context = nn.Linear(hidden_size * 2, 1, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        emb = self.dropout(self.embedding(x))
        h, _ = self.gru(emb)  # [B, W, 2H]

        u = torch.tanh(self.attn(h))
        a = self.context(u).squeeze(-1)  # [B, W]

        word_mask = (x != 0)
        a = a.masked_fill(~word_mask, -1e9)

        alpha = torch.softmax(a, dim=1)
        s = torch.sum(h * alpha.unsqueeze(-1), dim=1)  # [B, 2H]
        return s


class SentenceAttention(nn.Module):
    def __init__(self, input_size, hidden_size, dropout):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
            bidirectional=True,
        )
        self.attn = nn.Linear(hidden_size * 2, hidden_size * 2)
        self.context = nn.Linear(hidden_size * 2, 1, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, sents, sent_mask, return_attention=False):
        h, _ = self.gru(self.dropout(sents))
        u = torch.tanh(self.attn(h))
        a = self.context(u).squeeze(-1)

        mask = sent_mask > 0
        a = a.masked_fill(~mask, -1e9)

        alpha = torch.softmax(a, dim=1)
        v = torch.sum(h * alpha.unsqueeze(-1), dim=1)

        if return_attention:
            return v, alpha
        return v


class HAN(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim,
        word_hidden_size,
        sent_hidden_size,
        num_classes,
        dropout,
        embedding_matrix=None,
    ):
        super().__init__()
        self.word_encoder = WordAttention(
            vocab_size=vocab_size,
            embed_dim=embed_dim,
            hidden_size=word_hidden_size,
            dropout=dropout,
            embedding_matrix=embedding_matrix,
        )
        self.sent_encoder = SentenceAttention(
            input_size=word_hidden_size * 2,
            hidden_size=sent_hidden_size,
            dropout=dropout,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(sent_hidden_size * 2, num_classes),
        )

    def forward(self, doc_ids, sent_mask, return_attention=False):
        B, S, W = doc_ids.shape
        x = doc_ids.view(B * S, W)
        sent_vecs = self.word_encoder(x)
        sent_vecs = sent_vecs.view(B, S, -1)
    
        if return_attention:
            doc_vecs, sent_alpha = self.sent_encoder(sent_vecs, sent_mask, return_attention=True)
            logits = self.classifier(doc_vecs)
            return logits, sent_alpha

        doc_vecs = self.sent_encoder(sent_vecs, sent_mask)
        logits = self.classifier(doc_vecs)
        return logits

In [ ]:
model = HAN(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    word_hidden_size=WORD_HIDDEN_SIZE,
    sent_hidden_size=SENT_HIDDEN_SIZE,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT,
    embedding_matrix=embedding_matrix,
).to(DEVICE)

print(model)

In [ ]:
def evaluate(model, dataloader):
    model.eval()
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for batch in dataloader:
            doc_ids = batch["doc_ids"].to(DEVICE)
            sent_mask = batch["sent_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            logits = model(doc_ids, sent_mask)
            preds = torch.argmax(logits, dim=1)

            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    cm = confusion_matrix(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, output_dict=True)

    return {
        "accuracy": acc,
        "f1": f1,
        "confusion_matrix": cm,
        "classification_report": report,
    }

In [ ]:
def train_model(model, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    best_val_f1 = -math.inf
    best_state_dict = None
    best_epoch = None
    patience_counter = 0
    history = []

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        total_loss = 0.0

        for batch in tqdm(train_loader, desc=f"Training epoch {epoch}"):
            doc_ids = batch["doc_ids"].to(DEVICE)
            sent_mask = batch["sent_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            optimizer.zero_grad()
            logits = model(doc_ids, sent_mask)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)
        val_metrics = evaluate(model, val_loader)

        record = {
            "epoch": epoch,
            "train_loss": avg_train_loss,
            "val_accuracy": val_metrics["accuracy"],
            "val_f1": val_metrics["f1"],
        }
        history.append(record)

        print(f"\nEpoch {epoch}")
        print(f"Train loss: {avg_train_loss:.4f}")
        print(f"Validation Accuracy: {val_metrics['accuracy']:.4f}")
        print(f"Validation F1:       {val_metrics['f1']:.4f}")
        print("Validation CM:")
        print(val_metrics["confusion_matrix"])

        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print("Early stopping triggered.")
            break

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model, history, best_epoch, best_val_f1

In [ ]:
model, history, best_epoch, best_val_f1 = train_model(model, train_loader, val_loader)

print("\nBest validation epoch:", best_epoch)
print("Best validation F1:", round(best_val_f1, 4))

In [ ]:
test_metrics = evaluate(model, test_loader)

print("\n=== TEST PERFORMANCE ===")
print(f"Test Accuracy: {test_metrics['accuracy']:.4f}")
print(f"Test F1:       {test_metrics['f1']:.4f}")
print("Confusion Matrix:")
print(test_metrics["confusion_matrix"])

In [ ]:
torch.save(model.state_dict(), MODEL_PATH)

with open(VOCAB_PATH, "w", encoding="utf-8") as f:
    json.dump(vocab, f)

history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_PATH, index=False)

payload = {
    "dataset_tag": DATASET_TAG,
    "experiment_tag": EXPERIMENT_TAG,
    "run_tag": RUN_TAG,
    "device": DEVICE,
    "use_word2vec_init": bool(USE_WORD2VEC_INIT and WORD2VEC_KV_PATH.exists()),
    "vocab_size": len(vocab),
    "max_sentences": MAX_SENTENCES,
    "max_words_per_sentence": MAX_WORDS_PER_SENTENCE,
    "embed_dim": EMBED_DIM,
    "word_hidden_size": WORD_HIDDEN_SIZE,
    "sent_hidden_size": SENT_HIDDEN_SIZE,
    "dropout": DROPOUT,
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "best_validation_epoch": best_epoch,
    "best_validation_f1": best_val_f1,
    "test_results": {
        "test_accuracy": test_metrics["accuracy"],
        "test_f1": test_metrics["f1"],
        "confusion_matrix": test_metrics["confusion_matrix"].tolist(),
        "classification_report": test_metrics["classification_report"],
    },
}

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)

print("Saved:")
print(MODEL_PATH)
print(VOCAB_PATH)
print(HISTORY_PATH)
print(REPORT_PATH)

In [ ]:
payload

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

def preprocess_document_for_display(text):
    sentences = split_into_sentences(text)[:MAX_SENTENCES]
    display_sentences = []
    for sent in sentences:
        sent = sent.strip()
        if sent:
            display_sentences.append(sent)
    if not display_sentences:
        display_sentences = [""]
    return display_sentences

def encode_single_document(text, vocab):
    doc_tokens = preprocess_document(text)
    doc_ids, sent_mask = encode_document(doc_tokens, vocab)

    doc_ids = torch.tensor(doc_ids, dtype=torch.long).unsqueeze(0).to(DEVICE)
    sent_mask = torch.tensor(sent_mask, dtype=torch.float).unsqueeze(0).to(DEVICE)
    return doc_ids, sent_mask

doc_idx = 0
raw_text = test_df.iloc[doc_idx][TEXT_COL]
true_label = int(test_df.iloc[doc_idx][LABEL_COL])

display_sentences = preprocess_document_for_display(raw_text)
doc_ids, sent_mask = encode_single_document(raw_text, vocab)

model.eval()
with torch.no_grad():
    logits, sent_alpha = model(doc_ids, sent_mask, return_attention=True)
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    sent_alpha = sent_alpha.cpu().numpy()[0]

valid_len = min(len(display_sentences), len(sent_alpha))
display_sentences = display_sentences[:valid_len]
sent_alpha = sent_alpha[:valid_len]

pred_label = int(np.argmax(probs))

print("True label:", true_label)
print("Pred label:", pred_label)
print("Probabilities:", probs)
print("Number of sentences:", valid_len)

In [ ]:
plt.figure(figsize=(12, 4))
plt.bar(range(valid_len), sent_alpha)
plt.xlabel("Sentence index")
plt.ylabel("Attention weight")
plt.title(f"Sentence-level attention for document {doc_idx}")
plt.show()

In [ ]:
plt.figure(figsize=(12, 1.8))
plt.imshow(sent_alpha.reshape(1, -1), aspect="auto")
plt.yticks([])
plt.xlabel("Sentence index")
plt.title(f"Sentence attention heatmap for document {doc_idx}")
plt.colorbar(label="Attention")
plt.show()

In [ ]:
top_k = 5
top_indices = np.argsort(sent_alpha)[::-1][:top_k]

for rank, idx in enumerate(top_indices, start=1):
    print(f"\nTop {rank} sentence | attention = {sent_alpha[idx]:.4f}")
    print(display_sentences[idx])